# Route Tracer

Use this notebook to answer:
- **Which dates cover similar areas?** — bounding-box overlap and a similarity matrix
- **Which set of CSVs covers the same area with the longest combined time span?** — top 3 sets exported to CSV
- **Which single recording spans the longest time?** — duration ranking table




## 1) Configuration

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
DATA_DIR = PROJECT_ROOT / "Artifacts/BC_data"

MAP_LON_MIN, MAP_LON_MAX = None, None
MAP_LAT_MIN, MAP_LAT_MAX = None, None

MIN_STEP_M = 25.0
MAX_POINTS = 400
NETWORK_BUFFER_M = 400
USE_BASEMAP = True

OVERLAP_THRESHOLD = 0.80
MAX_GROUP_SIZE = 7
TOP_N_SETS = 3

RANKING_OUTPUT_CSV = DATA_DIR / "route_overlap_rankings.csv"
TOP_SETS_OUTPUT_CSV = DATA_DIR / "route_top_coverage_sets.csv"

ROUTE_COLORS = ["#e74c3c", "#2980b9", "#27ae60", "#8e44ad", "#d35400", "#16a085", "#c0392b", "#2c3e50"]



## 2) Helpers

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
from shapely.geometry import LineString, MultiLineString, box
from shapely.ops import linemerge

try:
    import contextily as cx
except ImportError:
    cx = None

from tqdm.auto import tqdm

REQUIRED_COLUMNS = ("datetime_local", "lat", "lon")

@dataclass
class TrackRecord:
    csv_path: Path
    label: str
    date_str: str
    time_label: str
    duration_hours: float
    gps_df: pd.DataFrame
    road_geom: object
    method: str
    coverage_box: object

def discover_csv_files(data_dir: Path) -> List[Path]:
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")
    return files

def load_track_csv(csv_path: Path) -> pd.DataFrame:
    path = Path(csv_path)
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} must contain {REQUIRED_COLUMNS}; missing: {missing}")

    out = df.copy()
    out["datetime_local"] = pd.to_datetime(out["datetime_local"], errors="coerce")
    out["lat"] = pd.to_numeric(out["lat"], errors="coerce")
    out["lon"] = pd.to_numeric(out["lon"], errors="coerce")
    out = out.dropna(subset=["datetime_local", "lat", "lon"])
    out = out[(out["lat"].between(-90, 90)) & (out["lon"].between(-180, 180))]
    out = out.sort_values("datetime_local").reset_index(drop=True)
    if out.empty:
        raise ValueError(f"{path.name}: no valid rows with datetime_local, lat, lon.")
    return out

def format_time_range_label(datetime_series: pd.Series) -> str:
    ts = pd.to_datetime(datetime_series, errors="coerce").dropna()
    start, end = ts.min(), ts.max()
    if start.normalize() == end.normalize():
        return f"{start.strftime('%Y-%m-%d')}  {start.strftime('%H:%M:%S')} – {end.strftime('%H:%M:%S')} local"
    return f"{start.strftime('%Y-%m-%d %H:%M:%S')} – {end.strftime('%Y-%m-%d %H:%M:%S')} local"

def track_duration_hours(datetime_series: pd.Series) -> float:
    ts = pd.to_datetime(datetime_series, errors="coerce").dropna()
    return (ts.max() - ts.min()).total_seconds() / 3600.0

def gps_coverage_box(df: pd.DataFrame):
    return box(df["lon"].min(), df["lat"].min(), df["lon"].max(), df["lat"].max())

def bbox_iou(a, b) -> float:
    inter = a.intersection(b).area
    union = a.union(b).area
    return inter / union if union > 0 else 0.0

def haversine_meters(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    return 2 * r * math.asin(math.sqrt(min(1.0, a)))

def simplify_track(df: pd.DataFrame, *, min_step_m: float = 25.0, max_points: Optional[int] = 400) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    keep_idx: List[int] = [0]
    last = df.iloc[0]
    for i in range(1, len(df)):
        row = df.iloc[i]
        if haversine_meters(last["lat"], last["lon"], row["lat"], row["lon"]) >= min_step_m:
            keep_idx.append(i)
            last = row
    if keep_idx[-1] != len(df) - 1:
        keep_idx.append(len(df) - 1)
    slim = df.iloc[keep_idx].reset_index(drop=True)
    if max_points is not None and len(slim) > max_points:
        positions = np.linspace(0, len(slim) - 1, max_points, dtype=int)
        slim = slim.iloc[sorted(set(positions))].reset_index(drop=True)
    return slim

def _collapse_consecutive(values: Sequence[int]) -> List[int]:
    if not values:
        return []
    out = [values[0]]
    for v in values[1:]:
        if v != out[-1]:
            out.append(v)
    return out

def trace_on_drive_network(lats: Iterable[float], lons: Iterable[float], *, buffer_m: float = 400.0):
    lat_arr = np.asarray(list(lats), dtype=float)
    lon_arr = np.asarray(list(lons), dtype=float)
    if len(lat_arr) < 2:
        geom = LineString([(lon_arr[0], lat_arr[0])]) if len(lat_arr) else LineString()
        return geom, "single_point"

    north, south = float(lat_arr.max()), float(lat_arr.min())
    east, west = float(lon_arr.max()), float(lon_arr.min())
    mid_lat = (north + south) / 2
    buf_lat = buffer_m / 111_000.0
    buf_lon = buffer_m / (111_000.0 * max(math.cos(math.radians(mid_lat)), 0.2))
    bbox = (west - buf_lon, south - buf_lat, east + buf_lon, north + buf_lat)

    G = ox.graph_from_bbox(bbox, network_type="drive", simplify=True)
    nodes = ox.nearest_nodes(G, lon_arr, lat_arr)
    node_seq = _collapse_consecutive(nodes.tolist() if hasattr(nodes, "tolist") else list(nodes))

    lines = []
    for a, b in zip(node_seq[:-1], node_seq[1:]):
        if a == b:
            continue
        try:
            route_nodes = ox.shortest_path(G, a, b, weight="length")
            if route_nodes is None:
                raise ValueError("no path")
            edge_gdf = ox.routing.route_to_gdf(G, route_nodes, weight="length")
            for geom in edge_gdf["geometry"]:
                if geom is not None and not geom.is_empty:
                    lines.append(geom)
        except Exception:
            lines.append(
                LineString([
                    (G.nodes[a]["x"], G.nodes[a]["y"]),
                    (G.nodes[b]["x"], G.nodes[b]["y"]),
                ])
            )

    if not lines:
        return LineString(list(zip(lon_arr, lat_arr))), "gps_fallback"
    return linemerge(lines), "osm_drive_network"

def build_track_record(csv_path: Path, color_idx: int) -> TrackRecord:
    gps_df = load_track_csv(csv_path)
    time_label = format_time_range_label(gps_df["datetime_local"])
    duration = track_duration_hours(gps_df["datetime_local"])
    date_str = gps_df["datetime_local"].iloc[0].strftime("%Y-%m-%d")
    start_t = gps_df["datetime_local"].iloc[0].strftime("%H:%M")
    end_t = gps_df["datetime_local"].iloc[-1].strftime("%H:%M")
    label = f"{date_str} ({start_t}–{end_t} local, {duration:.1f} h)"

    slim = simplify_track(gps_df, min_step_m=MIN_STEP_M, max_points=MAX_POINTS)
    road_geom, method = trace_on_drive_network(slim["lat"], slim["lon"], buffer_m=NETWORK_BUFFER_M)

    return TrackRecord(
        csv_path=csv_path,
        label=label,
        date_str=date_str,
        time_label=time_label,
        duration_hours=duration,
        gps_df=gps_df,
        road_geom=road_geom,
        method=method,
        coverage_box=gps_coverage_box(gps_df),
    )

def tracks_summary_table(tracks: List[TrackRecord]) -> pd.DataFrame:
    rows = []
    for t in tracks:
        df = t.gps_df
        rows.append({
            "CSV file": t.csv_path.name,
            "Date": t.date_str,
            "Start (local)": df["datetime_local"].iloc[0].strftime("%H:%M:%S"),
            "End (local)": df["datetime_local"].iloc[-1].strftime("%H:%M:%S"),
            "Duration (hours)": round(t.duration_hours, 2),
            "GPS points": len(df),
            "Lat min": round(df["lat"].min(), 5),
            "Lat max": round(df["lat"].max(), 5),
            "Lon min": round(df["lon"].min(), 5),
            "Lon max": round(df["lon"].max(), 5),
            "Map-match method": t.method,
        })
    return pd.DataFrame(rows).sort_values("Duration (hours)", ascending=False).reset_index(drop=True)

def coverage_similarity_matrix(tracks: List[TrackRecord]) -> pd.DataFrame:
    labels = [t.date_str for t in tracks]
    n = len(tracks)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            mat[i, j] = bbox_iou(tracks[i].coverage_box, tracks[j].coverage_box)
    return pd.DataFrame(mat, index=labels, columns=labels).round(3)

def apply_map_extent(ax, tracks=None):
    """Apply configured map bounds, or auto-fit to track coverage when unset."""
    if None not in (MAP_LON_MIN, MAP_LON_MAX, MAP_LAT_MIN, MAP_LAT_MAX):
        ax.set_xlim(MAP_LON_MIN, MAP_LON_MAX)
        ax.set_ylim(MAP_LAT_MIN, MAP_LAT_MAX)
    elif tracks:
        lons, lats = [], []
        for track in tracks:
            df = track.gps_df
            lons.extend([df["lon"].min(), df["lon"].max()])
            lats.extend([df["lat"].min(), df["lat"].max()])
        if lons and lats:
            pad_lon = max((max(lons) - min(lons)) * 0.05, 0.01)
            pad_lat = max((max(lats) - min(lats)) * 0.05, 0.01)
            ax.set_xlim(min(lons) - pad_lon, max(lons) + pad_lon)
            ax.set_ylim(min(lats) - pad_lat, max(lats) + pad_lat)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")

def plot_study_routes(tracks: List[TrackRecord], *, use_basemap: bool = True, figsize=(14, 12)):
    use_basemap = use_basemap and cx is not None
    fig, ax = plt.subplots(figsize=figsize)

    for i, track in enumerate(tracks):
        color = ROUTE_COLORS[i % len(ROUTE_COLORS)]
        geom = track.road_geom
        if geom is not None and not geom.is_empty:
            geoms = geom.geoms if isinstance(geom, MultiLineString) else [geom]
            for g in geoms:
                xs, ys = g.xy
                ax.plot(xs, ys, color=color, linewidth=2.5, solid_capstyle="round", zorder=4)
        ax.plot([], [], color=color, linewidth=2.5, label=track.label)

    apply_map_extent(ax, tracks)

    if use_basemap:
        cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron, zorder=1)
        ax.set_facecolor("none")
    else:
        ax.set_facecolor("#f7f7f7")
        ax.grid(True, linestyle=":", alpha=0.5)

    ax.set_title(
        "Vehicle routes on study-region road network\n(all recordings overlaid)",
        fontsize=14,
        pad=14,
    )
    ax.legend(title="Recording (start–end local, duration)", loc="upper left", framealpha=0.92, fontsize=9)
    fig.tight_layout()
    return fig

def pairwise_route_comparison(
    track_a: TrackRecord,
    track_b: TrackRecord,
) -> Tuple[pd.DataFrame, pd.DataFrame, float]:
    """Return per-track info, comparison metrics, and bounding-box route overlap."""
    overlap = bbox_iou(track_a.coverage_box, track_b.coverage_box)
    combined_tod = combined_time_of_day_metrics([track_a, track_b], [0, 1])
    combined_cal = combined_calendar_metrics([track_a, track_b], [0, 1])

    per_track = pd.DataFrame([
        {
            "Track": "A",
            "CSV file": track_a.csv_path.name,
            "Date": track_a.date_str,
            "Start (local)": track_a.gps_df["datetime_local"].iloc[0].strftime("%H:%M:%S"),
            "End (local)": track_a.gps_df["datetime_local"].iloc[-1].strftime("%H:%M:%S"),
            "Duration (hours)": round(track_a.duration_hours, 2),
            "Time range (local)": track_a.time_label,
        },
        {
            "Track": "B",
            "CSV file": track_b.csv_path.name,
            "Date": track_b.date_str,
            "Start (local)": track_b.gps_df["datetime_local"].iloc[0].strftime("%H:%M:%S"),
            "End (local)": track_b.gps_df["datetime_local"].iloc[-1].strftime("%H:%M:%S"),
            "Duration (hours)": round(track_b.duration_hours, 2),
            "Time range (local)": track_b.time_label,
        },
    ])

    comparison = pd.DataFrame([
        {"Metric": "Route overlap (bounding box IoU)", "Value": round(overlap, 4)},
        {"Metric": "Route overlap", "Value": f"{overlap:.1%}"},
        {"Metric": f"Same area (≥ {OVERLAP_THRESHOLD:.0%} overlap)", "Value": overlap >= OVERLAP_THRESHOLD},
        {"Metric": "Combined time-of-day start (local)", "Value": combined_tod["start"]},
        {"Metric": "Combined time-of-day end (local)", "Value": combined_tod["end"]},
        {"Metric": "Combined time-of-day span (hours)", "Value": round(combined_tod["span_hours"], 2)},
        {"Metric": "Combined calendar start (local)", "Value": combined_cal["start"]},
        {"Metric": "Combined calendar end (local)", "Value": combined_cal["end"]},
        {"Metric": "Combined calendar span (hours)", "Value": round(combined_cal["span_hours"], 2)},
        {"Metric": "Sum recording hours (hours)", "Value": round(track_a.duration_hours + track_b.duration_hours, 2)},
    ])

    return per_track, comparison, overlap

def plot_pairwise_routes(
    track_a: TrackRecord,
    track_b: TrackRecord,
    *,
    route_overlap: float,
    use_basemap: bool = True,
    figsize: Tuple[float, float] = (14, 12),
):
    use_basemap = use_basemap and cx is not None
    fig, ax = plt.subplots(figsize=figsize)

    for track, color in ((track_a, ROUTE_COLORS[0]), (track_b, ROUTE_COLORS[1])):
        geom = track.road_geom
        if geom is not None and not geom.is_empty:
            geoms = geom.geoms if isinstance(geom, MultiLineString) else [geom]
            for g in geoms:
                xs, ys = g.xy
                ax.plot(xs, ys, color=color, linewidth=2.5, solid_capstyle="round", zorder=4)
        ax.plot([], [], color=color, linewidth=2.5, label=track.label)

    apply_map_extent(ax, [track_a, track_b])

    if use_basemap:
        cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron, zorder=1)
        ax.set_facecolor("none")
    else:
        ax.set_facecolor("#f7f7f7")
        ax.grid(True, linestyle=":", alpha=0.5)

    ax.set_title(
        f"Pairwise route comparison — overlap {route_overlap:.1%}\n"
        f"{track_a.csv_path.name} vs {track_b.csv_path.name}",
        fontsize=13,
        pad=14,
    )
    ax.legend(title="Recording (start–end local, duration)", loc="upper left", framealpha=0.92, fontsize=9)
    fig.tight_layout()
    return fig

def _similarity_groups(tracks: List[TrackRecord], sim: pd.DataFrame, threshold: float = 0.85):
    """Union-find clusters of dates with similar GPS coverage."""
    n = len(tracks)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sim.iloc[i, j] >= threshold:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())

def _subset_pairwise_overlaps(indices: Sequence[int], sim) -> List[float]:
    if len(indices) < 2:
        return [1.0]
    if isinstance(sim, np.ndarray):
        sub = sim[np.ix_(indices, indices)]
        iu = np.triu_indices(len(indices), k=1)
        return sub[iu].tolist()
    return [sim.iloc[i, j] for i in range(len(indices)) for j in range(i + 1, len(indices))]

def _time_to_minutes(t) -> float:
    return t.hour * 60 + t.minute + t.second / 60.0

def _minutes_to_time_str(minutes: float) -> str:
    minutes = max(0.0, minutes)
    h = int(minutes // 60)
    m = int(minutes % 60)
    s = int(round((minutes * 60) % 60))
    if s == 60:
        s = 0
        m += 1
    if m == 60:
        m = 0
        h += 1
    return f"{h:02d}:{m:02d}:{s:02d}"

def combined_time_of_day_metrics(tracks: List[TrackRecord], indices: Sequence[int]) -> dict:
    """Clock-time envelope (local) when stitching same-area recordings from different dates."""
    start_minutes = min(_time_to_minutes(tracks[i].gps_df["datetime_local"].iloc[0].time()) for i in indices)
    end_minutes = max(_time_to_minutes(tracks[i].gps_df["datetime_local"].iloc[-1].time()) for i in indices)
    span_hours = (end_minutes - start_minutes) / 60.0
    return {
        "start": _minutes_to_time_str(start_minutes),
        "end": _minutes_to_time_str(end_minutes),
        "span_hours": span_hours,
    }

def combined_calendar_metrics(tracks: List[TrackRecord], indices: Sequence[int]) -> dict:
    starts = [tracks[i].gps_df["datetime_local"].iloc[0] for i in indices]
    ends = [tracks[i].gps_df["datetime_local"].iloc[-1] for i in indices]
    start, end = min(starts), max(ends)
    span_hours = (end - start).total_seconds() / 3600.0
    return {
        "start": start.strftime("%Y-%m-%d %H:%M:%S"),
        "end": end.strftime("%Y-%m-%d %H:%M:%S"),
        "span_hours": span_hours,
    }

def _build_group_row(
    tracks: List[TrackRecord],
    sim_arr: np.ndarray,
    indices: Sequence[int],
    *,
    area_cluster: int,
) -> dict:
    overlaps = _subset_pairwise_overlaps(indices, sim_arr)
    min_overlap = min(overlaps)
    tod = combined_time_of_day_metrics(tracks, indices)
    cal = combined_calendar_metrics(tracks, indices)
    return {
        "Area cluster": area_cluster,
        "N CSVs": len(indices),
        "CSV files": "; ".join(tracks[i].csv_path.name for i in indices),
        "Dates": "; ".join(tracks[i].date_str for i in indices),
        "Min pairwise overlap": round(min_overlap, 4),
        "Mean pairwise overlap": round(float(np.mean(overlaps)), 4),
        "Combined time-of-day start (local)": tod["start"],
        "Combined time-of-day end (local)": tod["end"],
        "Combined time-of-day span (hours)": round(tod["span_hours"], 2),
        "Combined calendar start (local)": cal["start"],
        "Combined calendar end (local)": cal["end"],
        "Combined calendar span (hours)": round(cal["span_hours"], 2),
        "Sum recording hours (hours)": round(sum(tracks[i].duration_hours for i in indices), 2),
        "Total GPS points": sum(len(tracks[i].gps_df) for i in indices),
    }

def rank_overlapping_csv_groups(
    tracks: List[TrackRecord],
    sim: pd.DataFrame,
    *,
    overlap_threshold: float = 0.85,
    max_group_size: int = 10,
) -> pd.DataFrame:
    """Rank same-area CSV groups (size 1..max_group_size) via clique search within overlap clusters."""
    sim_arr = sim.to_numpy()
    max_group_size = max(1, min(max_group_size, len(tracks)))
    rows = []
    clusters = _similarity_groups(tracks, sim, threshold=overlap_threshold)

    for cluster_id, comp in enumerate(tqdm(clusters, desc="Ranking same-area CSV groups", unit="cluster"), start=1):
        comp = sorted(comp)
        max_k = min(max_group_size, len(comp))

        def extend(clique: List[int], start_pos: int) -> None:
            if clique:
                rows.append(_build_group_row(tracks, sim_arr, clique, area_cluster=cluster_id))
            if len(clique) >= max_k:
                return
            for pos in range(start_pos, len(comp)):
                candidate = comp[pos]
                if all(sim_arr[candidate, member] >= overlap_threshold for member in clique):
                    extend(clique + [candidate], pos + 1)

        extend([], 0)

    if not rows:
        return pd.DataFrame()

    ranked = pd.DataFrame(rows).sort_values(
        [
            "Combined time-of-day span (hours)",
            "N CSVs",
            "Mean pairwise overlap",
            "Sum recording hours (hours)",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    ranked.insert(0, "Rank", ranked.index + 1)
    return ranked

def select_top_coverage_sets(rankings: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    """Best CSV combination per area cluster, then return the top n by combined time coverage."""
    if rankings.empty:
        return rankings.copy()

    best_per_cluster = (
        rankings.sort_values(
            ["Combined time-of-day span (hours)", "Sum recording hours (hours)", "N CSVs"],
            ascending=[False, False, False],
        )
        .groupby("Area cluster", as_index=False)
        .first()
    )
    top = (
        best_per_cluster.sort_values(
            ["Combined time-of-day span (hours)", "Sum recording hours (hours)"],
            ascending=[False, False],
        )
        .head(n)
        .reset_index(drop=True)
    )
    top.insert(0, "Set", range(1, len(top) + 1))
    top["Total time coverage"] = (
        top["Combined time-of-day start (local)"]
        + " – "
        + top["Combined time-of-day end (local)"]
        + " local ("
        + top["Combined time-of-day span (hours)"].astype(str)
        + " h daily window; "
        + top["Sum recording hours (hours)"].astype(str)
        + " h recorded)"
    )
    display_cols = [
        "Set",
        "Area cluster",
        "N CSVs",
        "CSV files",
        "Dates",
        "Combined time-of-day start (local)",
        "Combined time-of-day end (local)",
        "Combined time-of-day span (hours)",
        "Sum recording hours (hours)",
        "Mean pairwise overlap",
        "Total time coverage",
    ]
    return top[display_cols]

def print_comparison_insights(
    tracks: List[TrackRecord],
    summary: pd.DataFrame,
    sim: pd.DataFrame,
    rankings: pd.DataFrame,
    top_sets: pd.DataFrame,
):
    longest = summary.iloc[0]
    print("LONGEST SINGLE RECORDING")
    print(f"  {longest['CSV file']} — {longest['Duration (hours)']} hours on {longest['Date']}")
    print()

    if not top_sets.empty:
        print(f"TOP {len(top_sets)} SAME-AREA CSV SETS (by combined time coverage)")
        for _, row in top_sets.iterrows():
            print(f"  Set {row['Set']} — {row['CSV files']}")
            print(f"    Total time coverage: {row['Total time coverage']}")
            print(f"    Mean overlap: {row['Mean pairwise overlap']:.1%}")
        print()
    elif not rankings.empty:
        best = rankings.iloc[0]
        print("BEST SAME-AREA CSV SET (longest combined time-of-day span)")
        print(f"  {best['CSV files']}")
        print(
            f"  Time-of-day coverage: {best['Combined time-of-day start (local)']} – "
            f"{best['Combined time-of-day end (local)']} local ({best['Combined time-of-day span (hours)']} h)"
        )
        print()

    print(f"SIMILAR AREA PAIRS (bounding-box overlap ≥ {OVERLAP_THRESHOLD:.0%})")
    dates = [t.date_str for t in tracks]
    found = False
    for i in range(len(dates)):
        for j in range(i + 1, len(dates)):
            score = sim.iloc[i, j]
            if score >= OVERLAP_THRESHOLD:
                found = True
                print(f"  {dates[i]} ↔ {dates[j]}: overlap = {score:.1%}")
    if not found:
        print(f"  No pairs above {OVERLAP_THRESHOLD:.0%} overlap — see the similarity matrix for partial overlap.")
    print()

    print("AREA CLUSTERS (union of all high-overlap CSVs)")
    groups = _similarity_groups(tracks, sim, threshold=OVERLAP_THRESHOLD)
    for group in groups:
        member_labels = ", ".join(tracks[i].date_str for i in sorted(group))
        csv_names = ", ".join(tracks[i].csv_path.name for i in sorted(group))
        print(f"  [{member_labels}] → {csv_names}")
    print()

    print("HOW TO READ THIS")
    print("  • Section 7 shows the top 3 same-area CSV sets with total time coverage.")
    print("  • Combined time-of-day span = earliest start to latest end across selected CSVs (e.g. 06:00–18:00 local).")
    print("  • Sum recording hours = total hours of readings in that area (adds across different dates).")
    

## 3) Discover CSV files

In [ ]:
csv_files = discover_csv_files(DATA_DIR)
print(f"Found {len(csv_files)} CSV file(s) in {DATA_DIR}:")
for p in csv_files:
    print(f"  • {p.name}")

## 4) Map-match all tracks

Each CSV is simplified, snapped to the OSM drive network, and stored for plotting. The first run downloads road data (network access required).

In [ ]:
tracks: List[TrackRecord] = []
for i, csv_path in enumerate(csv_files):
    print(f"Processing {csv_path.name} ...")
    tracks.append(build_track_record(csv_path, i))
    print(f"  done — {tracks[-1].label} [{tracks[-1].method}]")
print(f"\nProcessed {len(tracks)} track(s).")

## 5) Summary — duration and spatial extent

Sorted by **duration (hours)** descending. Use this to find the longest recording.

In [ ]:
summary = tracks_summary_table(tracks)
summary

## 6) Coverage similarity matrix

Values are **bounding-box overlap** (0 = no shared area, 1 = identical extent). High values mean the vehicle covered a similar part of the study region on those dates.

In [ ]:
similarity = coverage_similarity_matrix(tracks)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.heatmap(similarity, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.5, cbar_kws={'label': 'Bounding Box Overlap'})
plt.title("Coverage Similarity Matrix (Bounding Box Overlap)")
plt.xlabel("Date")
plt.ylabel("Date")
plt.tight_layout()
plt.show()

## 7) Same-area CSV sets — top 3 by combined time coverage

Groups of **1 to `MAX_GROUP_SIZE` CSVs** (default 5) where all pairs overlap at least `OVERLAP_THRESHOLD` are ranked by **combined time-of-day span**. The **top `TOP_N_SETS` sets** (default 3) — one best group per area cluster — are shown below with total time coverage.

Outputs:
- `route_top_coverage_sets.csv` — top 3 recommended sets
- `route_overlap_rankings.csv` — full ranking of all valid groups

In [ ]:
overlap_rankings = rank_overlapping_csv_groups(
    tracks,
    similarity,
    overlap_threshold=OVERLAP_THRESHOLD,
    max_group_size=MAX_GROUP_SIZE,
)

if overlap_rankings.empty:
    top_coverage_sets = overlap_rankings
    print(f"No CSV groups met the overlap threshold ({OVERLAP_THRESHOLD:.0%}). Lower OVERLAP_THRESHOLD in section 1.")
else:
    top_coverage_sets = select_top_coverage_sets(overlap_rankings, n=TOP_N_SETS)
    multi = overlap_rankings[overlap_rankings["N CSVs"] >= 2]
    print(f"Found {len(overlap_rankings)} same-area group(s); {len(multi)} with 2+ CSVs.")
    print(f"Top {len(top_coverage_sets)} set(s) saved to: {TOP_SETS_OUTPUT_CSV}")
    print(f"Full ranking ({len(overlap_rankings)} rows) saved to: {RANKING_OUTPUT_CSV}\n")
    top_coverage_sets.to_csv(TOP_SETS_OUTPUT_CSV, index=False)
    overlap_rankings.to_csv(RANKING_OUTPUT_CSV, index=False)
    top_coverage_sets

## 8) Insights

In [ ]:
print_comparison_insights(tracks, summary, similarity, overlap_rankings, top_coverage_sets)

## 9) Study-region map — all routes overlaid

Map extent uses configured bounds, or auto-fits loaded tracks. Each colour is one recording date; legend shows date, local time range, and duration.

In [ ]:
fig = plot_study_routes(tracks, use_basemap=USE_BASEMAP)
plt.show()

## 10) Manual pairwise comparison (2 CSVs)

Pick any two CSV files and compare them directly: **route overlap**, **time ranges**, and **road traces** on the study-region map.

Edit the two paths below, then run the cell.

In [ ]:
from IPython.display import display

PAIR_CSV_A = DATA_DIR / "BC_video_1s_YYYYMMDD_a.csv"
PAIR_CSV_B = DATA_DIR / "BC_video_1s_YYYYMMDD_b.csv"

print("Track A:", PAIR_CSV_A)
print("Track B:", PAIR_CSV_B)

print("\nMap-matching track A ...")
pair_track_a = build_track_record(PAIR_CSV_A, 0)
print(f"  done — {pair_track_a.label}")

print("Map-matching track B ...")
pair_track_b = build_track_record(PAIR_CSV_B, 1)
print(f"  done — {pair_track_b.label}")

pair_tracks_info, pair_comparison, pair_overlap = pairwise_route_comparison(pair_track_a, pair_track_b)

print("\n--- Time ranges ---")
display(pair_tracks_info)
print("--- Route overlap & combined coverage ---")
display(pair_comparison)

fig = plot_pairwise_routes(
    pair_track_a,
    pair_track_b,
    route_overlap=pair_overlap,
    use_basemap=USE_BASEMAP,
)
plt.show()

